In [ ]:
"""
تجربة مباشرة بالكاميرا - ASL 10 words (نسخة XY + Z للأصابع = 374 feature)
الملفات المطلوبة بنفس المجلد: fold0.pth fold1.pth fold2.pth fold3.pth
تثبيت: pip install torch opencv-python mediapipe numpy pandas pyarrow
خروج: q   |   تصفير الـ buffer: r
"""
import glob
import os
import warnings
from collections import deque

import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

# ===================== نفس إعدادات التدريب (لا تغيّرها) =====================
WORDS = ['TV', 'apple', 'bird', 'book', 'car', 'cat', 'chair', 'clean', 'cloud', 'clown']
T = 48
LH_OFF, POSE_OFF, RH_OFF = 468, 489, 522
LIPS = [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
        78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308]
POSE_SEL = [POSE_OFF + i for i in (11, 12, 13, 14, 15, 16)]
SEL_XY = np.array(LIPS + list(range(LH_OFF, LH_OFF + 21)) + list(range(RH_OFF, RH_OFF + 21)) + POSE_SEL)

NOSE_IDX = POSE_OFF + 0
LH_TIPS = [LH_OFF + i for i in (4, 8, 12, 16, 20)]
RH_TIPS = [RH_OFF + i for i in (4, 8, 12, 16, 20)]
SEL_Z = np.array([NOSE_IDX] + LH_TIPS + RH_TIPS)

N_PTS_XY, N_PTS_Z = len(SEL_XY), len(SEL_Z)       # 88, 11
IN_DIM = (N_PTS_XY * 2 + N_PTS_Z) * 2             # 374

# ===================== إعدادات التجربة =====================
BUF_LEN = 70            # عدد الفريمات اللي بنخزنها (~2.3 ثانية)
MIN_HAND_FRAMES = 12    # أقل عدد فريمات لازم تظهر فيها اليد عشان نتنبأ
THRESHOLD = 0.70        # أقل من هيك = ما بعرض كلمة
SMOOTH = 5              # متوسط آخر N تنبؤات
PREDICT_EVERY = 2       # تنبؤ كل N فريم
NO_HAND_RESET = 20      # إذا اختفت اليدين هالعدد من الفريمات، بنفضي الـ buffer

# اختياري: اختبر الموديل على ملف parquet من الداتا قبل فتح الكاميرا (للتأكد إن كل شي سليم)
# مثال: TEST_PARQUET = r"C:\data\train_landmark_files\16069\100015657.parquet"; TEST_LABEL = "cloud"
TEST_PARQUET = None
TEST_LABEL = None

MODEL_DIR = os.path.dirname(os.path.abspath("best_model\fold0.pth"))


# ===================== الموديل (نفس الهيكل بالتدريب بالضبط) =====================
class ConvBlock(nn.Module):
    def __init__(self, d, k, drop):
        super().__init__()
        self.conv = nn.Conv1d(d, d, k, padding=k // 2)
        self.bn = nn.BatchNorm1d(d)
        self.act = nn.GELU()
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        h = self.drop(self.act(self.bn(self.conv(x.transpose(1, 2))))).transpose(1, 2)
        return x + h


class SignNet(nn.Module):
    def __init__(self, in_dim=IN_DIM, n_cls=len(WORDS), d=192, heads=4, layers=2, drop=0.3):
        super().__init__()
        self.embed = nn.Sequential(nn.Linear(in_dim, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(drop))
        self.convs = nn.ModuleList([ConvBlock(d, k, drop) for k in (5, 5, 3)])
        self.pos = nn.Parameter(torch.zeros(1, T, d))
        enc = nn.TransformerEncoderLayer(d, heads, d * 2, drop, batch_first=True,
                                         norm_first=True, activation='gelu')
        self.tr = nn.TransformerEncoder(enc, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(2 * d), nn.Dropout(drop), nn.Linear(2 * d, n_cls))

    def forward(self, x):
        x = self.embed(x)
        for c in self.convs:
            x = c(x)
        x = self.tr(x + self.pos)
        x = torch.cat([x.mean(1), x.max(1).values], -1)
        return self.head(x)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
paths = sorted(glob.glob(os.path.join(MODEL_DIR, 'fold*.pth')))
if not paths:
    raise FileNotFoundError("ما لقيت fold*.pth - حطهم بنفس مجلد السكربت")
models = []
for p in paths:
    m = SignNet().to(device)
    m.load_state_dict(torch.load(p, map_location=device))
    models.append(m.eval())
print(f"تم تحميل {len(models)} موديلات:", [os.path.basename(p) for p in paths])


# ===================== نفس معالجة التدريب (load_sequence + make_feats) =====================
def preprocess(arr):
    """arr: (n, 543, 3) = x,y,z وفيها NaN للمفقود -> (T, 374)"""
    arr_xy = arr[..., :2].copy()
    arr_z = arr[..., 2:3].copy()

    # قص الفريمات الفاضية
    hands = np.concatenate([arr_xy[:, LH_OFF:LH_OFF + 21, 0], arr_xy[:, RH_OFF:RH_OFF + 21, 0]], axis=1)
    valid = np.where(~np.isnan(hands).all(axis=1))[0]
    if len(valid) >= 2:
        arr_xy = arr_xy[valid[0]:valid[-1] + 1]
        arr_z = arr_z[valid[0]:valid[-1] + 1]

    # التطبيع: المركز = منتصف الكتفين، المقياس = عرض الكتفين
    sh = arr_xy[:, [POSE_OFF + 11, POSE_OFF + 12]]
    center = np.nanmean(sh.reshape(-1, 2), axis=0)
    width = np.nanmean(np.linalg.norm(sh[:, 0] - sh[:, 1], axis=-1))

    if np.isnan(center).any() or np.isnan(width) or width < 1e-3:
        allp = arr_xy[:, SEL_XY].reshape(-1, 2)
        center = np.nanmean(allp, axis=0) if not np.isnan(allp).all() else np.zeros(2)
        width = 2 * np.nanstd(allp) if not np.isnan(allp).all() else 1.0
        width = width if width >= 1e-3 else 1.0

    pts_xy = (arr_xy[:, SEL_XY] - center) / width

    nose_z = arr_z[:, NOSE_IDX:NOSE_IDX + 1, :]
    nose_mean = np.nanmean(nose_z)
    nose_z = np.where(np.isnan(nose_z), np.nan_to_num(nose_mean), nose_z)
    pts_z = (arr_z[:, SEL_Z] - nose_z) / width

    pts_xy = pts_xy.reshape(len(pts_xy), -1)
    pts_z = pts_z.reshape(len(pts_z), -1)
    pts = np.concatenate([pts_xy, pts_z], axis=-1)

    idx = np.linspace(0, len(pts) - 1, T).round().astype(int)
    x = np.nan_to_num(pts[idx], nan=0.0).astype(np.float32)      # (T, 187)

    # make_feats: الأصلية + الدلتا
    m = (x == 0).all(-1)
    inv = (m[1:] | m[:-1])[..., None]
    d = np.zeros_like(x)
    d[1:] = np.where(inv, 0, x[1:] - x[:-1])
    return np.concatenate([x, d], axis=-1).astype(np.float32)    # (T, 374)


@torch.no_grad()
def predict(arr):
    feats = torch.from_numpy(preprocess(arr)).unsqueeze(0).to(device)
    assert feats.shape[-1] == IN_DIM, feats.shape
    return np.mean([torch.softmax(m(feats), -1).cpu().numpy()[0] for m in models], axis=0)


# ===================== اختبار على ملف من الداتا (اختياري) =====================
if TEST_PARQUET:
    import pandas as pd
    d = pd.read_parquet(TEST_PARQUET, columns=['x', 'y', 'z']).to_numpy(np.float32)
    n = len(d) // 543
    P = predict(d[:n * 543].reshape(n, 543, 3))
    top = np.argsort(-P)[:3]
    print("اختبار parquet:", " | ".join(f"{WORDS[i]} {P[i]:.2f}" for i in top),
          "| المتوقع:", TEST_LABEL)

# ===================== MediaPipe =====================
mp_holistic = mp.solutions.holistic
mp_draw = mp.solutions.drawing_utils


def extract(results):
    """نفس ترتيب الداتا: face | left_hand | pose | right_hand، والمفقود = NaN. (543, 3)"""
    a = np.full((543, 3), np.nan, np.float32)
    if results.face_landmarks:
        a[:468] = [[l.x, l.y, l.z] for l in results.face_landmarks.landmark]
    if results.left_hand_landmarks:
        a[LH_OFF:LH_OFF + 21] = [[l.x, l.y, l.z] for l in results.left_hand_landmarks.landmark]
    if results.pose_landmarks:
        a[POSE_OFF:POSE_OFF + 33] = [[l.x, l.y, l.z] for l in results.pose_landmarks.landmark]
    if results.right_hand_landmarks:
        a[RH_OFF:RH_OFF + 21] = [[l.x, l.y, l.z] for l in results.right_hand_landmarks.landmark]
    return a


def hand_visible(a):
    return not (np.isnan(a[LH_OFF:LH_OFF + 21, 0]).all() and np.isnan(a[RH_OFF:RH_OFF + 21, 0]).all())


# ===================== الحلقة الرئيسية =====================
buffer = deque(maxlen=BUF_LEN)
recent = deque(maxlen=SMOOTH)
word, conf_shown = "", 0.0
frame_i, no_hand = 0, 0

cap = cv2.VideoCapture(0)
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        # مهم: ما نعمل flip للصورة قبل المعالجة (بيبدّل اليمين باليسار وبخرّب التنبؤ)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = holistic.process(rgb)

        a = extract(results)
        buffer.append(a)
        frame_i += 1

        if hand_visible(a):
            no_hand = 0
        else:
            no_hand += 1
            if no_hand >= NO_HAND_RESET:
                buffer.clear(); recent.clear(); word = ""

        hand_frames = sum(hand_visible(b) for b in buffer)
        if hand_frames >= MIN_HAND_FRAMES and frame_i % PREDICT_EVERY == 0:
            recent.append(predict(np.stack(buffer)))
            P = np.mean(recent, axis=0)
            i = int(P.argmax())
            if P[i] >= THRESHOLD:
                word, conf_shown = WORDS[i], float(P[i])
            else:
                word = ""

        for lm, conn in [(results.pose_landmarks, mp_holistic.POSE_CONNECTIONS),
                         (results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS),
                         (results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)]:
            if lm:
                mp_draw.draw_landmarks(frame, lm, conn)
        cv2.rectangle(frame, (0, 0), (frame.shape[1], 60), (245, 117, 16), -1)
        txt = f"{word}  {conf_shown*100:.0f}%" if word else "..."
        cv2.putText(frame, txt, (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255, 255, 255), 3, cv2.LINE_AA)
        cv2.putText(frame, f"hand frames: {hand_frames}/{BUF_LEN}", (20, frame.shape[0] - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)

        cv2.imshow('ASL 10 words', frame)
        k = cv2.waitKey(1) & 0xFF
        if k == ord('q'):
            break
        if k == ord('r'):
            buffer.clear(); recent.clear(); word = ""

cap.release()
cv2.destroyAllWindows()